# 🦋 Butterfly Image Classification — CNN
### Multiclass image classification using EfficientNetB0 + Flask API
**Dataset:** [Butterfly Image Classification — Kaggle](https://www.kaggle.com/datasets/phucthaiv02/butterfly-image-classification)

---
**Notebook flow:**
1. Install & import
2. Load dataset
3. Explore data (EDA)
4. Preprocess & augment
5. Build CNN model (EfficientNetB0)
6. Train the model
7. Evaluate (accuracy, confusion matrix)
8. Predict on a single image
9. Export model artifacts
10. 🌐 Flask API — serve predictions


## 1. Install & Import Libraries

In [ ]:
# Install if running on Kaggle or fresh environment
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

print("PyTorch version:", torch.__version__)
print("Device:", "GPU ✅" if torch.cuda.is_available() else "CPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Load Dataset
> **Folder structure expected:**
> ```
> /butterfly-image-classification/
>     train/   (images)
>     test/    (images)
>     Training_set.csv   (filename, label)
>     Testing_set.csv    (filename)
> ```

In [ ]:

BASE_DIR = "/kaggle/input/datasets/proggadipannita/butterfly/archive" # Kaggle path

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
TRAIN_CSV = os.path.join(BASE_DIR, "Training_set.csv")
TEST_CSV = os.path.join(BASE_DIR, "Testing_set.csv")
# ── Load CSV ──────────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train samples:", len(train_df))
print("Test  samples:", len(test_df))
print()
print(train_df.head())


## 3. Explore Data (EDA)

In [ ]:
# ── Class names & counts ──────────────────────────────────────────────────────
class_counts = train_df['label'].value_counts()
num_classes  = len(class_counts)
print(f"Total classes: {num_classes}")
print(f"Min samples per class: {class_counts.min()} ({class_counts.idxmin()})")
print(f"Max samples per class: {class_counts.max()} ({class_counts.idxmax()})")

# ── Bar chart ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(20, 5))
class_counts.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title(f'Class Distribution — {num_classes} butterfly species', fontsize=14)
plt.xlabel('Species')
plt.ylabel('Number of images')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# ── Show sample images ────────────────────────────────────────────────────────
sample_classes = random.sample(list(class_counts.index), min(10, num_classes))

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for ax, cls in zip(axes, sample_classes):
    # get a random image for this class
    sample_row = train_df[train_df['label'] == cls].sample(1).iloc[0]
    img_path   = os.path.join(TRAIN_DIR, sample_row['filename'])
    img        = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(cls, fontsize=8, pad=4)
    ax.axis('off')

plt.suptitle('Sample images per class', fontsize=13)
plt.tight_layout()
plt.show()


## 4. Preprocess & Augment

In [ ]:
# ── Encode string labels → integers ──────────────────────────────────────────
le = LabelEncoder()
train_df['label_id'] = le.fit_transform(train_df['label'])
class_names = list(le.classes_)
print("Classes:", class_names[:5], "... total:", len(class_names))

# ── Train / validation split (80/20) ─────────────────────────────────────────
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df, test_size=0.2, stratify=train_df['label_id'], random_state=42
)
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")


In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
IMG_SIZE = 224   # EfficientNetB0 input size
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("Transforms ready ✅")


In [ ]:
# ── Custom PyTorch Dataset ────────────────────────────────────────────────────
class ButterflyDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        image    = Image.open(img_path).convert("RGB")
        label    = row['label_id']
        if self.transform:
            image = self.transform(image)
        return image, label

# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE = 32

train_dataset = ButterflyDataset(train_data, TRAIN_DIR, train_transform)
val_dataset   = ButterflyDataset(val_data,   TRAIN_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 5. Build CNN Model (EfficientNetB0 Transfer Learning)

In [ ]:
# ── Fallback: load without pretrained weights (no internet needed) ────────────
model = models.efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features  # 1280

model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes),
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
# ── Load pretrained EfficientNetB0 ───────────────────────────────────────────
#weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
#model   = models.efficientnet_b0(weights=weights)
model = models.efficientnet_b0(weights=None)
# ── Replace the classifier head ───────────────────────────────────────────────
in_features = model.classifier[1].in_features   # 1280

model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes),
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


## 6. Train the Model
Two-phase training:
- **Phase 1** (epochs 1–3): freeze backbone, only train the new head
- **Phase 2** (epochs 4–N): unfreeze everything and fine-tune

In [ ]:
# ── Freeze backbone for Phase 1 ───────────────────────────────────────────────
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)

EPOCHS       = 10
PHASE2_EPOCH = 4      # unfreeze at this epoch

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):

    # ── Switch to Phase 2 ─────────────────────────────────────────────────────
    if epoch == PHASE2_EPOCH:
        print(f"\n--- Epoch {epoch}: Unfreezing backbone (Phase 2) ---\n")
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        t_loss    += loss.item() * images.size(0)
        t_correct += (outputs.argmax(1) == labels).sum().item()
        t_total   += images.size(0)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            v_loss    += loss.item() * images.size(0)
            v_correct += (outputs.argmax(1) == labels).sum().item()
            v_total   += images.size(0)

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    train_losses.append(t_loss / t_total)
    val_losses.append(v_loss / v_total)
    train_accs.append(t_acc)
    val_accs.append(v_acc)

    print(f"Epoch {epoch:>2}/{EPOCHS} | "
          f"Train Loss: {t_loss/t_total:.4f}  Acc: {t_acc:.4f} | "
          f"Val Loss: {v_loss/v_total:.4f}  Acc: {v_acc:.4f}")

    # ── Save best model ───────────────────────────────────────────────────────
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print(f"  ✅ Best model saved (val_acc={v_acc:.4f})")

print(f"\nTraining complete. Best Val Acc: {best_val_acc:.4f}")


In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_losses, label='Train', marker='o')
axes[0].plot(val_losses,   label='Val',   marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(train_accs, label='Train', marker='o')
axes[1].plot(val_accs,   label='Val',   marker='o')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Evaluate the Model

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds  = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# ── Classification report ─────────────────────────────────────────────────────
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig_size = max(10, num_classes * 0.4)
plt.figure(figsize=(fig_size, fig_size))
sns.heatmap(
    cm_norm,
    xticklabels=class_names,
    yticklabels=class_names,
    cmap='Blues',
    annot=num_classes <= 20,
    fmt='.2f' if num_classes <= 20 else '',
    linewidths=0.3,
)
plt.title('Confusion Matrix (Normalized)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.show()


## 8. Predict on a Single Image

In [ ]:
def predict_image(img_path, top_k=5):
    """Run the CNN on one image and return top-k predictions."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])

    img    = Image.open(img_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze()

    top_probs, top_idx = torch.topk(probs, k=top_k)

    print(f"\n📸  Image: {os.path.basename(img_path)}")
    print(f"{'─'*40}")
    for i, (prob, idx) in enumerate(zip(top_probs, top_idx)):
        marker = "►" if i == 0 else " "
        print(f" {marker} {i+1}. {class_names[idx]:<35} {prob*100:>6.2f}%")
    print(f"{'─'*40}")

    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Predicted: {class_names[top_idx[0]]} ({top_probs[0]*100:.1f}%)")
    plt.axis('off')
    plt.show()

    return class_names[top_idx[0]], float(top_probs[0])

# ── Test on a random validation image ─────────────────────────────────────────
sample      = val_data.sample(1).iloc[0]
sample_path = os.path.join(TRAIN_DIR, sample['filename'])
predicted_class, confidence = predict_image(sample_path)
print(f"\nActual label : {sample['label']}")
print(f"Predicted    : {predicted_class}")


In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/best_model.pth')

## 9. Export Model & Labels for the Flask API

To serve the model outside this notebook we need three artifacts saved next to `app.py`:

- `best_model.pth` — the trained weights (already saved during training)
- `class_names.json` — the ordered list of butterfly species names (index → label)
- `config.json` — preprocessing settings (image size, normalization mean/std) so the API uses the exact same pipeline as training/inference above


In [ ]:
import json

# ── Save class names (index -> label mapping, in the order the model outputs them) ──
with open("class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

# ── Save preprocessing config so the API matches training exactly ──────────────
config = {
    "img_size": IMG_SIZE,
    "mean": MEAN,
    "std": STD,
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved {len(class_names)} class names to class_names.json")
print("Saved preprocessing config to config.json")
print("\nMake sure 'best_model.pth', 'class_names.json' and 'config.json' are all in the same folder as app.py")


## 10. 🌐 Flask API

This section generates a small **Flask** application (`app.py`) that loads the trained
EfficientNetB0 model and exposes a REST endpoint for image classification, plus a
simple HTML upload page for quick manual testing.

**Endpoints:**
- `GET  /` — simple HTML page to upload an image and see the prediction
- `GET  /health` — health check, returns model/class info
- `POST /predict` — send an image file (form field `file`) → returns JSON with top-k predictions

**Files generated:**
- `app.py` — the Flask server
- `requirements.txt` — Python dependencies
- `templates/index.html` — upload form for the `/` page

**To run it** (after running the export cell above so `best_model.pth`, `class_names.json`
and `config.json` exist in this folder):

```bash
pip install -r requirements.txt
python app.py
```

Then open **http://127.0.0.1:5000** in your browser, or call the API directly:

```bash
curl -X POST -F "file=@/path/to/butterfly.jpg" "http://127.0.0.1:5000/predict?top_k=5"
```


In [ ]:
%%writefile app.py
"""
Flask API for the Butterfly Image Classification model (EfficientNetB0 + PyTorch).

Run with:
    python app.py

Then:
    - Open http://127.0.0.1:5000 in a browser for a simple upload UI
    - Or POST an image to /predict, e.g.:
        curl -X POST -F "file=@image.jpg" "http://127.0.0.1:5000/predict?top_k=5"
"""

import io
import json
import os

import torch
import torch.nn as nn
from flask import Flask, jsonify, render_template, request
from PIL import Image
from torchvision import models, transforms

# ---------------------------------------------------------------------------
# Config & artifacts
# ---------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

MODEL_PATH       = os.path.join(BASE_DIR, "best_model.pth")
CLASS_NAMES_PATH = os.path.join(BASE_DIR, "class_names.json")
CONFIG_PATH      = os.path.join(BASE_DIR, "config.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(CLASS_NAMES_PATH, "r") as f:
    class_names = json.load(f)
num_classes = len(class_names)

# Fall back to the training defaults if config.json wasn't generated
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)
else:
    config = {"img_size": 224, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}

IMG_SIZE = config["img_size"]
MEAN     = config["mean"]
STD      = config["std"]

# ---------------------------------------------------------------------------
# Build model architecture (must match training) and load weights
# ---------------------------------------------------------------------------
model = models.efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features  # 1280

model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes),
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# ---------------------------------------------------------------------------
# Preprocessing (same as predict_image() in the notebook)
# ---------------------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ---------------------------------------------------------------------------
# Flask app
# ---------------------------------------------------------------------------
app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 10 * 1024 * 1024  # 10 MB upload limit


def predict_image_bytes(image_bytes, top_k=5):
    """Run the model on raw image bytes and return top-k (label, probability) pairs."""
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze(0)

    top_k = max(1, min(top_k, num_classes))
    top_probs, top_idx = torch.topk(probs, k=top_k)

    return [
        {"class": class_names[idx], "confidence": round(float(prob), 6)}
        for prob, idx in zip(top_probs, top_idx)
    ]


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "device": str(device),
        "num_classes": num_classes,
        "img_size": IMG_SIZE,
    })


@app.route("/predict", methods=["POST"])
def predict():
    if "file" not in request.files:
        return jsonify({"error": "No file part in the request. Use form field 'file'."}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "No file selected."}), 400

    try:
        top_k = int(request.args.get("top_k", 5))
    except ValueError:
        top_k = 5

    try:
        image_bytes = file.read()
        results = predict_image_bytes(image_bytes, top_k=top_k)
    except Exception as e:
        return jsonify({"error": f"Could not process image: {e}"}), 400

    return jsonify({
        "filename": file.filename,
        "predicted_class": results[0]["class"],
        "confidence": results[0]["confidence"],
        "top_k": results,
    })


if __name__ == "__main__":
    # Set debug=False in production
    app.run(host="0.0.0.0", port=5000, debug=True)


In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Butterfly Classifier</title>
  <style>
    body { font-family: Arial, sans-serif; max-width: 560px; margin: 40px auto; padding: 0 16px; }
    h1 { font-size: 1.5rem; }
    .card { border: 1px solid #ddd; border-radius: 10px; padding: 20px; margin-top: 16px; }
    button { background: #2563eb; color: #fff; border: none; padding: 10px 18px;
             border-radius: 6px; cursor: pointer; font-size: 1rem; }
    button:hover { background: #1d4ed8; }
    #result { margin-top: 16px; white-space: pre-wrap; font-family: monospace; }
    img#preview { max-width: 100%; margin-top: 12px; border-radius: 8px; display: none; }
  </style>
</head>
<body>
  <h1>🦋 Butterfly Species Classifier</h1>
  <div class="card">
    <form id="form">
      <input type="file" id="file" accept="image/*" required>
      <br><br>
      <button type="submit">Predict</button>
    </form>
    <img id="preview" alt="preview">
    <div id="result"></div>
  </div>

  <script>
    const form = document.getElementById('form');
    const fileInput = document.getElementById('file');
    const preview = document.getElementById('preview');
    const result = document.getElementById('result');

    fileInput.addEventListener('change', () => {
      const file = fileInput.files[0];
      if (file) {
        preview.src = URL.createObjectURL(file);
        preview.style.display = 'block';
      }
    });

    form.addEventListener('submit', async (e) => {
      e.preventDefault();
      const file = fileInput.files[0];
      if (!file) return;

      result.textContent = 'Predicting...';

      const formData = new FormData();
      formData.append('file', file);

      try {
        const res = await fetch('/predict?top_k=5', { method: 'POST', body: formData });
        const data = await res.json();
        if (data.error) {
          result.textContent = 'Error: ' + data.error;
          return;
        }
        let text = `Predicted: ${data.predicted_class} (${(data.confidence * 100).toFixed(2)}%)\n\nTop predictions:\n`;
        data.top_k.forEach((p, i) => {
          text += `${i + 1}. ${p.class} - ${(p.confidence * 100).toFixed(2)}%\n`;
        });
        result.textContent = text;
      } catch (err) {
        result.textContent = 'Request failed: ' + err;
      }
    });
  </script>
</body>
</html>


In [ ]:
%%writefile requirements.txt
flask
torch
torchvision
pillow
